# Chapter 3. 베이즈 정리, 그리고 생성 vs 판별 — 실습 노트북

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SuminHan/book-ml/blob/main/notebooks/ml1/chapter03_1_bayes.ipynb)

책 본문: [3.1 베이즈 정리, 그리고 생성 vs 판별](https://smhanlab.com/book-ml/kor/ml1/chapter03/1.html)

이 노트북은 책 3.1절의 스팸 필터 베이즈 정리 계산을 코드로 재현해서
**본문의 손 계산값이 실제로 맞는지** 직접 검증합니다. 순차적 갱신과
일괄 곱셈의 동등성, 로그-오즈(log-odds) 형태, 의료 검진 예(기저율
오류), 그리고 "로그를 더해야 하는" 수치적 이유(언더플로우)까지
끝까지 실행됩니다.

## 1. 본문 손 계산 바로 검증 (스팸 필터)

본문 설정: 전체 메일의 30%가 스팸, "무료"는 스팸의 60%, 정상 메일의
5%에 등장 → "무료"가 포함된 메일의 스팸 사후확률은 약 83.7%여야
합니다.

In [1]:
prior_spam = 0.3
p_word_given_spam = 0.6
p_word_given_ham = 0.05

numer = p_word_given_spam * prior_spam                 # 0.18
denom = numer + p_word_given_ham * (1 - prior_spam)    # 0.215
posterior = numer / denom

print(f"분자   = {numer:.4f}")
print(f"분모   = {denom:.4f}")
print(f'P(spam|"무료") = {posterior:.4f}')
assert abs(posterior - 0.8372) < 1e-4
print("본문 손 계산(약 0.8372)과 정확히 일치")

분자   = 0.1800
분모   = 0.2150
P(spam|"무료") = 0.8372
본문 손 계산(약 0.8372)과 정확히 일치


## 2. 순차적 갱신 vs 일괄 곱셈 (본문의 `bayes_update`)

본문 "실습: 여러 증거를 곱해서 갱신하기"의 코드를 그대로 가져왔습니다.
모든 증거를 **일괄** 곱하는 것과 증거를 **하나씩 순서대로** 갱신하는
것은 결과가 완전히 같아야 합니다 — 이것이 베이즈 정리가 "순차적
추론의 틀"이라는 뜻입니다.

In [2]:
def bayes_update(prior_spam, likelihoods_spam, likelihoods_ham):
    # likelihoods_*: 관찰된 각 단어의 P(단어|클래스) 리스트
    p_spam = prior_spam
    for l_spam in likelihoods_spam:
        p_spam *= l_spam
    p_ham = 1 - prior_spam
    for l_ham in likelihoods_ham:
        p_ham *= l_ham
    return p_spam / (p_spam + p_ham)

# 일괄: "무료"(0.6 vs 0.05)와 "당첨"(0.4 vs 0.02)을 한꺼번에 관찰
all_at_once = bayes_update(0.3, [0.6, 0.4], [0.05, 0.02])

# 순차: "무료"로 갱신한 사후확률을 새로운 사전확률로 쓰고, "당첨"을 반영
after_free  = bayes_update(0.3, [0.6], [0.05])
after_prize = bayes_update(after_free, [0.4], [0.02])

print(f"일괄 계산:        {all_at_once:.4f}")
print(f"'무료' 이후:      {after_free:.4f}")
print(f"'당첨' 이후:      {after_prize:.4f}")
assert abs(all_at_once - after_prize) < 1e-12
print("일괄과 순차적 갱신이 정확히 같은 결과를 줌")

일괄 계산:        0.9904
'무료' 이후:      0.8372
'당첨' 이후:      0.9904
일괄과 순차적 갱신이 정확히 같은 결과를 줌


## 3. 로그-오즈 형태: "가능도를 곱한다" = "로그 가능도비를 더한다"

베이즈 정리를 **오즈**(odds, P/(1-P))로 다시 쓰면:

\[
\frac{P(y|x)}{P(\neg y|x)} = \frac{P(x|y)}{P(x|\neg y)} \cdot \frac{P(y)}{P(\neg y)}
\]

즉 **사후오즈 = 사전오즈 × 가능도비(likelihood ratio)** — 증거 하나당
오즈가 일정한 배수로 갱신됩니다. 로그를 취하면 곱셈이 덧셈이 되는데,
이것이 3.3절 스팸 필터가 "확률의 곱" 대신 "로그의 합"을 계산하는
이유입니다.

In [3]:
import math

def sigmoid(z):
    return 1.0 / (1.0 + math.exp(-z))

def log_odds_to_prob(log_odds):
    # P = odds / (1 + odds), odds = exp(log_odds)  ->  시그모이드와 동일
    return sigmoid(log_odds)

prior_log_odds = math.log(0.3 / 0.7)          # log(3/7) 약 -0.847
log_lr_free  = math.log(0.6 / 0.05)           # log 12  약 2.485
log_lr_prize = math.log(0.4 / 0.02)           # log 20  약 2.996

one = log_odds_to_prob(prior_log_odds + log_lr_free)
two = log_odds_to_prob(prior_log_odds + log_lr_free + log_lr_prize)

print(f"가능도비 로그: '무료' {log_lr_free:.3f}, '당첨' {log_lr_prize:.3f}")
print(f"'무료'만:  사후확률 = {one:.4f}")
print(f"두 단어:   사후확률 = {two:.4f}")
assert abs(one - 0.8372) < 1e-4
assert abs(two - 0.9904) < 1e-4
print("2절의 곱셈 계산 결과(0.8372 -> 0.9904)와 완벽히 일치")

가능도비 로그: '무료' 2.485, '당첨' 2.996
'무료'만:  사후확률 = 0.8372
두 단어:   사후확률 = 0.9904
2절의 곱셈 계산 결과(0.8372 -> 0.9904)와 완벽히 일치


## 4. 의료 검진 예: 기저율 오류 (본문 "자주 하는 실수")

유병률 0.1%, 민감도·특이도 99%인 검사에서 양성 판정을 받았을 때의
사후확률은 약 9%뿐이라는 본문의 계산을 코드로 확인하고, 유병률
(기저율)이 떨어질수록 사후확률이 어떻게 내려가는지 봅니다.

In [4]:
def p_disease_given_positive(base_rate, sens=0.99, spec=0.99):
    return sens * base_rate / (sens * base_rate + (1 - spec) * (1 - base_rate))

for base in [0.10, 0.01, 0.001, 0.0001]:
    print(f"유병률 {base*100:6.2f}%: P(병|양성) = {p_disease_given_positive(base):.4f}")

# "1000명 세어보기"로 재확인: 1명 병에 걸려 있고 999명 건강
tp = 1 * 0.99        # 진짜 양성 (true positive)
fp = 999 * 0.01      # 가짜 양성 (false positive)
print(f"1000명: 진짜 양성 {tp:.1f}명 vs 가짜 양성 {fp:.1f}명")
print(f"P(병|양성) = {tp:.1f} / {tp+fp:.1f} = {tp/(tp+fp):.4f}")
print("가짜 양성이 진짜 양성의 약 10배 — 이것이 기저율 오류")

유병률  10.00%: P(병|양성) = 0.9167
유병률   1.00%: P(병|양성) = 0.5000
유병률   0.10%: P(병|양성) = 0.0902
유병률   0.01%: P(병|양성) = 0.0098
1000명: 진짜 양성 1.0명 vs 가짜 양성 10.0명
P(병|양성) = 1.0 / 11.0 = 0.0902
가짜 양성이 진짜 양성의 약 10배 — 이것이 기저율 오류


## 5. 왜 로그를 더할 수밖에 없는가: 언더플로우 수치 확인

스팸 필터는 단어가 **수천~수만 개**인 경우, 1보다 작은 가능도를
거의 그대로 반복 곱해야 합니다. 부동소수점에서는 그 곱이 언젠가
**0.0으로 언더플로우**하고, 그 시점 이후에는 어떤 증거도 더 반영할
수 없게 됩니다. 로그의 합은 유한한 음수라서 이런 일이 없습니다.

In [5]:
import math

# 실제 스팸 필터처럼 어휘가 수천 개: 3000개 단어가 모두 가능도 0.1
n_words = 3000
likelihoods = [0.1] * n_words

prod = 1.0
for p in likelihoods:
    prod *= p                    # 0.1을 3000번 곱함
logsum = sum(math.log(p) for p in likelihoods)   # 로그를 3000번 더함

print(f"{n_words}개 가능도의 곱: {prod}")   # 0.0 — 부동소수점 언더플로우
print(f"로그의 합:             {logsum:.4f}")  # -6907.76 — 유한한 음수
assert prod == 0.0
assert abs(logsum - n_words * math.log(0.1)) < 1e-6
print("곱은 0.0으로 사라졌지만 로그의 합은 유한 — 비교는 언제나 로그의 합으로")

3000개 가능도의 곱: 0.0
로그의 합:             -6907.7553
곱은 0.0으로 사라졌지만 로그의 합은 유한 — 비교는 언제나 로그의 합으로


## 6. 시각화: 기저율이 사후확률을 얼마나 끌고 내려가는가

민감도·특이도 99% 검정의 사후확률을 **사전오즈**(pre-test odds)의
함수로 그림. 본문의 예(유병률 0.1% = 오즈 1/1000)와 10배 더 드문
경우(1/10000)를 표시합니다. 검사 성능(가능도비 99)은 오즈를
**일시적으로 99배** 올릴 뿐, 기저율이 낮으면 사후확률이 여전히
아주 낮다는 점을 log-log 그래프로 보여줍니다.

In [6]:
import numpy as np
import matplotlib
matplotlib.use("Agg")
# 한글 라벨 렌더링 (시스템에 CJK 폰트가 있으면 쓰고, 없으면 기본 폰트로 폴백)
matplotlib.rcParams["font.sans-serif"] = ["Noto Sans CJK KR", "NanumGothic", "DejaVu Sans"]
matplotlib.rcParams["axes.unicode_minus"] = False
import matplotlib.pyplot as plt

sens = spec = 0.99
odds = np.logspace(-4, -0.3, 300)   # 사전오즈: 1/10000 ~ 약 0.5
# 사후확률 = sens*o/(sens*o + (1-spec)) / (1+o)  (오즈 -> 확률 변환 포함)
post = sens * odds / (sens * odds + (1 - spec)) / (1 + odds)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(odds, post, lw=2, label="P(disease|positive)")
for o, lab, off in [(1/1000, "odds 1/1000 (prevalence 0.1%) \u2192 ~9%", (25, -5)),
                    (1/10000, "odds 1/10000 (prevalence 0.01%) \u2192 ~1%", (25, 5))]:
    y = sens * o / (sens * o + (1 - spec)) / (1 + o)
    ax.axvline(o, ls="--", c="gray", alpha=0.6)
    ax.plot([o], [y], "ro")
    ax.annotate(lab, (o, y), textcoords="offset points", xytext=off, fontsize=9)
ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlabel("Prior odds (pre-test odds)")
ax.set_ylabel("Posterior P(disease|positive)")
ax.set_title("Even a 99%-accurate test can't raise the posterior much when the base rate is low")
ax.grid(True, which="both", alpha=0.3)
ax.legend()
fig.tight_layout()
fig.savefig("fig_base_rate.png", dpi=120)
plt.show()
print("검사의 가능도비(99)는 오즈의 일회성 승수 — 기저율을 이겨낼 만큼 강하지 못하다")

검사의 가능도비(99)는 오즈의 일회성 승수 — 기저율을 이겨낼 만큼 강하지 못하다


## 7. 시각화: 생성적 모델이 어떻게 판별적 결과를 뽑아내는가

1차원 스팸 필터의 **전체 흐름**을 하나의 그림으로. 각 클래스(스팸/햄)가
데이터를 '만드는 법' \(P(x|y)\) — 여기서는 가우시안 가능도 — 을 배우고,
사전확률 0.3으로 가중한 뒤, 두 가중 곡선이 **교차하는 지점**이 결정
경계 \(x^*\!\approx\!1.27\)가 된다(사전이 0.5였다면 1.0이었을).
오른쪽 축의 초록 곡선은 그 교차 구조로부터 베이즈 정리로 '뒤집어'
얻은 사후확률 \(P(\text{spam}|x)\) — 3.2절 GDA가 바로 이 구조의
표준형이다.

In [7]:
import numpy as np
import matplotlib
matplotlib.use("Agg")
matplotlib.rcParams["font.sans-serif"] = ["Noto Sans CJK KR", "NanumGothic", "DejaVu Sans"]
matplotlib.rcParams["axes.unicode_minus"] = False
import matplotlib.pyplot as plt

p_spam = 0.3
mu_spam, sig_spam = 2.0, 0.8     # 스팸: '무료' 신호가 강한 쪽
mu_ham,  sig_ham  = 0.0, 0.8     # 햄: 신호 약한 쪽
def gauss(x, mu, s):
    return np.exp(-0.5*((x-mu)/s)**2)/(s*np.sqrt(2*np.pi))

x = np.linspace(-2.5, 4.5, 1200)
L_spam = gauss(x, mu_spam, sig_spam); L_ham = gauss(x, mu_ham, sig_ham)
W_spam = p_spam*L_spam; W_ham = (1-p_spam)*L_ham
post = W_spam/(W_spam+W_ham)

i = np.argmin(np.abs(W_spam-W_ham)); xstar = x[i]
fig, axL = plt.subplots(figsize=(8.4, 5.0))
axL.plot(x, L_spam, color="#e8912d", lw=2.2, label="Likelihood $P(x|\\text{spam})$")
axL.plot(x, L_ham,  color="#4a90d9", lw=2.2, label="Likelihood $P(x|\\text{ham})$")
axL.plot(x, W_spam, color="#e8912d", lw=1.6, ls="--", alpha=0.9, label="Prior-weighted $P(x|\\text{spam})P(\\text{spam})$")
axL.plot(x, W_ham,  color="#4a90d9", lw=1.6, ls="--", alpha=0.9, label="Prior-weighted $P(x|\\text{ham})P(\\text{ham})$")
axL.axvline(xstar, color="#212529", ls=":", lw=1.5)
axL.annotate(f"Decision boundary $x^*\\!\\approx\\!{xstar:.2f}$\n(prior 0.30 -> right of the midpoint 1.0)",
             xy=(xstar, 0.02), xytext=(xstar+0.15, 0.28), fontsize=9,
             color="#212529", arrowprops=dict(arrowstyle="->", color="#212529", lw=1.0))
axL.set_xlim(-2.5, 4.5); axL.set_ylim(0, 0.62)
axL.set_xlabel("Feature value $x$ (1D, e.g. strength of 'free' in an email)")
axL.set_ylabel("Density $P(x|y)$", color="#495057")
axL.grid(True, alpha=0.25)
axR = axL.twinx()
axR.plot(x, post, color="#2f9e44", lw=3.0, label="Posterior $P(\\text{spam}|x)$")
axR.set_ylim(0, 1); axR.set_ylabel("Posterior $P(y|x)$", color="#2f9e44")
axR.tick_params(axis="y", labelcolor="#2f9e44")
h1, l1 = axL.get_legend_handles_labels(); h2, l2 = axR.get_legend_handles_labels()
axL.legend(h1+h2, l1+l2, loc="upper left", fontsize=8.5, framealpha=0.9)
axL.set_title("Generative model: each class learns how it 'generates' data ($P(x|y)$),\nthen Bayes' rule inverts it to get the posterior $P(y|x)$")
fig.tight_layout()
fig.savefig("fig_generative_vs_discriminative.png", dpi=130)
fig.savefig("ch03_generative_vs_discriminative.svg")
plt.show()
print(f"결정 경계 x* ≈ {xstar:.3f} — 사전 0.30이 0.5보다 작으니 중점 1.0보다 오른쪽으로 이동")
print("P(spam|x=0) =", f"{gauss(0, mu_spam, sig_spam)*p_spam/(gauss(0, mu_spam, sig_spam)*p_spam + gauss(0, mu_ham, sig_ham)*(1-p_spam)):.4f}",
      ",  P(spam|x=2) =", f"{gauss(2, mu_spam, sig_spam)*p_spam/(gauss(2, mu_spam, sig_spam)*p_spam + gauss(2, mu_ham, sig_ham)*(1-p_spam)):.4f}")


결정 경계 x* ≈ 1.271 — 사전 0.30이 0.5보다 작으니 중점 1.0보다 오른쪽으로 이동
P(spam|x=0) = 0.0185 ,  P(spam|x=2) = 0.9070


## 정리: 다음으로

- **3.2절 (GDA)**: 가능도 \(P(x|y)\)에 **정규분포**를 둔 생성적 모델 —
  연속값 특징을 다룹니다.
- **3.3절 (나이브베이즈)**: 가능도에 **단어 독립 가정**을 둔 생성적 모델 —
  이 노트북의 `bayes_update`는 그 2클래스·2단어 축소판이고, 3.3절의
  실제 필터는 어휘 전체(수만 단어)에 대해 3절의 "로그의 합"을 수행합니다.

모든 셀이 에러 없이 끝까지 실행된다면, 본문 3.1절의 숫자 예시
(0.8372 → 0.9904, 약 9%, 기저율 표)가 전부 코드로 검증된 셈입니다.